# 도형 CNN: 랜덤 탐색과 시각화 (Colab Pro)

직육면체(cuboid)와 수직 원기둥(cylinder)의 **64×64 RGB** 이미지를 분류합니다. 사전학습 가중치는 사용하지 않습니다. 깊이는 이미지 영역을 찾을 때만 사용하고 CNN에는 넣지 않습니다.

1. Ubuntu에서 `scripts/collect_cnn_data.py`로 만든 zip을 Google Drive에 업로드합니다.
2. Colab 메뉴에서 **런타임 → 런타임 유형 변경 → GPU**를 선택합니다. Pro여도 GPU 종류·실행 가능 시간은 보장하지 않으므로 아래 설정을 줄일 수 있습니다.
3. 경로와 탐색 횟수를 바꾸고 위에서부터 실행합니다. **12회·최대20epoch·조기종료4epoch**가 기본입니다. 첫 확인은 2회·3epoch로 권장합니다.
4. 검증 macro F1로 모델을 고른 뒤, 최종 테스트는 마지막 셀에서 한 번만 평가합니다. 테스트 결과를 보고 하이퍼파라미터를 다시 고르면 테스트가 검증 데이터 역할을 하게 됩니다.

노트북 코드는 Git 관리합니다. 데이터·가중치·실행 결과는 로컬 `data/`, `checkpoints/`, `outputs/` 또는 개인 Drive에 보관합니다. Git에 올릴 때 출력은 지우세요.

In [ ]:
# Colab의 PyTorch는 그대로 사용하고 평가·시각화 도구를 준비합니다.
%pip -q install tqdm scikit-learn matplotlib pandas Pillow

In [ ]:
# Colab에서 개인 Drive를 연결합니다. Drive에 올린 zip 경로는 다음 셀에서 지정합니다.
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import copy
from datetime import datetime
import hashlib
import importlib.metadata
import json
import math
from pathlib import Path
import random
import shutil
import time
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image, ImageEnhance
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score,
                             roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay,
                             classification_report)

In [ ]:
# 경로만 본인 Drive 위치에 맞춥니다. GPU 사용 여부는 출력으로 확인하세요.
ZIP_PATH = Path('/content/drive/MyDrive/shape_cnn_v1.zip')
DATA_ROOT = Path('/content/shape_cnn_v1')
RUNS_ROOT = Path('/content/drive/MyDrive/shape_cnn_runs')
SEED = 2026
NUM_TRIALS = 12
MAX_EPOCHS = 20
PATIENCE = 4
BATCH_CHOICES = [32, 64, 128]
CLASSES = ['cuboid', 'cylinder']  # 0=직육면체, 1=원기둥: 저장·추론까지 같은 순서
MEAN = [0.5, 0.5, 0.5]
STD = [0.5, 0.5, 0.5]
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
RUN_DIR = RUNS_ROOT / datetime.now().strftime('%Y%m%dT%H%M%S_%f')
RUN_DIR.mkdir(parents=True, exist_ok=False)
assert NUM_TRIALS >= 1 and MAX_EPOCHS >= 1 and PATIENCE >= 1
print('device:', DEVICE)
if DEVICE.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('CPU로 실행합니다. Colab GPU 설정을 확인하거나 탐색 횟수를 줄이세요.')

In [ ]:
# 새 경로에 풀어 이전 데이터와 섞이지 않게 합니다. 재실행은 추출 없이 같은 파일을 검사합니다.
assert ZIP_PATH.is_file(), f'먼저 Drive에 데이터 zip을 올리세요: {ZIP_PATH}'
archive_sha = hashlib.sha256(ZIP_PATH.read_bytes()).hexdigest()
marker = DATA_ROOT / '.archive_sha256'
if DATA_ROOT.exists():
    assert marker.is_file() and marker.read_text().strip() == archive_sha, '다른 데이터가 있는 경로입니다. DATA_ROOT를 새 경로로 바꾸세요.'
else:
    DATA_ROOT.mkdir(parents=True)
    with zipfile.ZipFile(ZIP_PATH) as archive:
        for member in archive.infolist():
            assert (DATA_ROOT / member.filename).resolve().is_relative_to(DATA_ROOT.resolve()), '잘못된 zip 경로'
        archive.extractall(DATA_ROOT)
    marker.write_text(archive_sha)
dataset_config = json.loads((DATA_ROOT / 'config.json').read_text())
assert dataset_config['status'] == 'complete', '완료되지 않은 수집 데이터입니다.'
assert dataset_config['classes'] == CLASSES and dataset_config['crop_size'] == 64
manifest_bytes = (DATA_ROOT / 'manifest.jsonl').read_bytes()
assert hashlib.sha256(manifest_bytes).hexdigest() == dataset_config['manifest_sha256']
rows = [json.loads(line) for line in manifest_bytes.decode().splitlines()]
by_split = {split: [r for r in rows if r['split'] == split] for split in ('train', 'val', 'test')}
# 같은 장면·seed가 서로 다른 분할에 들어갔는지 검사합니다.
scene_splits, seed_splits, paths = {}, {}, set()
for row in tqdm(rows, desc='데이터 확인'):
    assert row['split'] in by_split and row['label'] in CLASSES
    for key, mapping in (('scene_id', scene_splits), ('seed', seed_splits)):
        assert mapping.setdefault(row[key], row['split']) == row['split'], '장면 분할 누출'
    assert row['path'] not in paths, '중복 이미지 경로'
    paths.add(row['path'])
    path = DATA_ROOT / row['path']
    assert path.resolve().is_relative_to(DATA_ROOT.resolve())
    assert Path(row['path']).parts[:2] == (row['split'], row['label'])
    with Image.open(path) as image:
        assert image.size == (64, 64) and image.mode == 'RGB'
for split, subset in by_split.items():
    assert {r['label'] for r in subset} == set(CLASSES), f'{split}에 두 클래스가 모두 필요합니다.'
counts = pd.crosstab(pd.Series([r['split'] for r in rows], name='split'),
                     pd.Series([r['label'] for r in rows], name='class')).reindex(['train', 'val', 'test'])
display(counts)
counts.plot.bar(title='Images in each split', rot=0)
plt.ylabel('images'); plt.tight_layout(); plt.savefig(RUN_DIR / 'dataset_counts.png'); plt.show()
# 가로 폭·높이 분포를 비교해 특정 크기만 한쪽 분할에 몰렸는지도 확인합니다.
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
for split, subset in by_split.items():
    axes[0].hist([max(r['size_m'][:-1]) * 100 for r in subset], bins=12, alpha=.4, label=split, density=True)
    axes[1].hist([r['size_m'][-1] * 100 for r in subset], bins=12, alpha=.4, label=split, density=True)
axes[0].set_title('Width (cm)'); axes[1].set_title('Height (cm)')
for ax in axes: ax.legend()
plt.tight_layout(); plt.savefig(RUN_DIR / 'size_distribution.png'); plt.show()
(RUN_DIR / 'dataset_config.json').write_text(json.dumps(dataset_config, indent=2))

## 이미지 처리와 모델

이미지를 0~1로 바꾸고 `(픽셀값−0.5)/0.5`로 정규화합니다. 검증·테스트에는 증강을 넣지 않습니다. 학습 때만 밝기·대비를 약간 바꾸며, 로컬 추론에도 같은 정규화를 적용해야 합니다.

작은 CNN은 Conv → ReLU → Pool을 세 번 거쳐 두 클래스 점수를 출력합니다. softmax는 **지표를 구할 때만** 적용합니다. 교차엔트로피에는 softmax 전 점수(logits)를 넣습니다.

In [ ]:
def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

class ShapeDataset(Dataset):
    def __init__(self, subset, augment=False):
        self.rows, self.augment = subset, augment
    def __len__(self): return len(self.rows)
    def __getitem__(self, index):
        row = self.rows[index]
        with Image.open(DATA_ROOT / row['path']) as raw:
            image = raw.convert('RGB')
        if self.augment:
            image = ImageEnhance.Brightness(image).enhance(random.uniform(.9, 1.1))
            image = ImageEnhance.Contrast(image).enhance(random.uniform(.9, 1.1))
        tensor = torch.from_numpy(np.asarray(image, dtype=np.float32).copy()).permute(2, 0, 1) / 255
        tensor = (tensor - torch.tensor(MEAN)[:, None, None]) / torch.tensor(STD)[:, None, None]
        return tensor, CLASSES.index(row['label'])

class SmallCNN(nn.Module):
    def __init__(self, dropout=0.15):
        super().__init__()
        self.features = nn.Sequential(nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                                      nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                                      nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                                      nn.AdaptiveAvgPool2d((4, 4)))
        self.classifier = nn.Sequential(nn.Flatten(), nn.Dropout(dropout), nn.Linear(64 * 4 * 4, 64),
                                        nn.ReLU(), nn.Linear(64, len(CLASSES)))
    def forward(self, x): return self.classifier(self.features(x))

def metrics(y, probabilities):
    predictions = probabilities.argmax(axis=1)
    return {'accuracy': float(accuracy_score(y, predictions)),
            'macro_f1': float(f1_score(y, predictions, labels=[0, 1], average='macro', zero_division=0)),
            'macro_precision': float(precision_score(y, predictions, labels=[0, 1], average='macro', zero_division=0)),
            'macro_recall': float(recall_score(y, predictions, labels=[0, 1], average='macro', zero_division=0)),
            # 이진 ROC-AUC는 원기둥(클래스1)의 확률로 계산합니다.
            'roc_auc': float(roc_auc_score(y, probabilities[:, 1])) if len(np.unique(y)) == 2 else None}

def evaluate(model, loader, criterion):
    model.eval()
    total_loss, n, labels, probs = 0., 0, [], []
    # 평가에서는 기울기를 계산하거나 가중치를 바꾸지 않습니다.
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            logits = model(x)
            total_loss += criterion(logits, y).item() * len(y); n += len(y)
            labels.extend(y.cpu().tolist()); probs.extend(logits.softmax(dim=1).cpu().tolist())
    y, probabilities = np.asarray(labels), np.asarray(probs)
    return {'loss': total_loss / n, **metrics(y, probabilities)}, y, probabilities

## 랜덤 하이퍼파라미터 탐색

첫 실험은 기준 설정, 나머지는 seed로 재현 가능한 랜덤 탐색입니다. 학습률은 로그 범위 `0.0001~0.01`에서 뽑습니다. 옵티마이저는 Adam/AdamW/SGD, 손실은 교차엔트로피/스무딩 교차엔트로피를 비교합니다. 레이블 스무딩은 정답을 100% 확신하도록만 학습시키지 않는 설정입니다.

**최고 검증 macro F1**의 epoch를 저장하고 동률이면 일반 교차엔트로피(NLL)가 낮은 쪽을 고릅니다. 손실 종류가 다르면 loss 숫자도 달라질 수 있으므로 실험 간 loss 비교에는 공통 NLL을 사용합니다. 테스트 데이터는 이 탐색에서 읽지 않습니다.

In [ ]:
def sample_trials(seed, count):
    rng = random.Random(seed)
    trials = [{'optimizer': 'AdamW', 'loss': 'cross_entropy', 'lr': 1e-3,
               'batch_size': 64, 'weight_decay': 1e-4, 'dropout': .15}]
    while len(trials) < count:
        params = {'optimizer': rng.choice(['Adam', 'AdamW', 'SGD']),
                  'loss': rng.choice(['cross_entropy', 'smoothed_cross_entropy']),
                  'lr': 10 ** rng.uniform(-4, -2), 'batch_size': rng.choice(BATCH_CHOICES),
                  'weight_decay': 10 ** rng.uniform(-6, -2), 'dropout': rng.choice([0., .15, .3])}
        if params not in trials: trials.append(params)
    return trials

def train_trial(params, trial_id):
    seed_everything(SEED)  # 설정 비교에서 시작 가중치/순서를 가능한 한 맞춥니다.
    trial_dir = RUN_DIR / f'trial_{trial_id:02d}'; trial_dir.mkdir(exist_ok=False)
    (trial_dir / 'params.json').write_text(json.dumps(params, indent=2))
    model = SmallCNN(params['dropout']).to(DEVICE)
    # 손실 설정을 바꾸되, 모델 출력은 두 클래스의 logits로 통일합니다.
    criterion = nn.CrossEntropyLoss(label_smoothing=.1 if params['loss'] == 'smoothed_cross_entropy' else 0.)
    nll_criterion = nn.CrossEntropyLoss()
    # 옵티마이저는 기울기로 가중치를 수정하는 규칙입니다.
    optimizer_class = {'Adam': torch.optim.Adam, 'AdamW': torch.optim.AdamW, 'SGD': torch.optim.SGD}[params['optimizer']]
    extras = {'momentum': .9} if params['optimizer'] == 'SGD' else {}
    optimizer = optimizer_class(model.parameters(), lr=params['lr'], weight_decay=params['weight_decay'], **extras)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=2, factor=.5)
    train_loader = DataLoader(ShapeDataset(by_split['train'], True), batch_size=params['batch_size'], shuffle=True,
                              generator=torch.Generator().manual_seed(SEED), num_workers=0, pin_memory=DEVICE.type == 'cuda')
    val_loader = DataLoader(ShapeDataset(by_split['val']), batch_size=128, shuffle=False, num_workers=0)
    best_key, best_epoch, stale, history = (-1., -math.inf), 0, 0, []
    start = time.monotonic()
    for epoch in tqdm(range(1, MAX_EPOCHS + 1), desc=f'Trial {trial_id} epochs', leave=False):
        model.train(); total_loss, count, correct = 0., 0, 0
        for x, y in tqdm(train_loader, desc=f'epoch {epoch} batches', leave=False):
            x, y = x.to(DEVICE), y.to(DEVICE)
            # 이전 배치의 기울기를 지우고 현재 배치의 손실로 가중치를 갱신합니다.
            optimizer.zero_grad(set_to_none=True)
            logits = model(x); loss = criterion(logits, y)
            loss.backward(); optimizer.step()
            total_loss += loss.item() * len(y); count += len(y)
            correct += int((logits.argmax(dim=1) == y).sum().item())
        val, y_val, probabilities = evaluate(model, val_loader, criterion)
        # 같은 예측 확률로 계산한 일반 NLL이 loss 종류에 상관없는 비교 기준입니다.
        nll = float(-np.log(np.clip(probabilities[np.arange(len(y_val)), y_val], 1e-12, 1)).mean())
        row = {'epoch': epoch, 'train_loss': total_loss / count, 'train_accuracy': correct / count,
               'lr': optimizer.param_groups[0]['lr'], **{f'val_{k}': v for k, v in val.items()}, 'val_nll': nll}
        history.append(row)
        pd.DataFrame(history).to_csv(trial_dir / 'history.csv', index=False)
        # 마지막 epoch가 아니라, 검증 결과가 가장 좋았던 epoch를 보관합니다.
        key = (val['macro_f1'], -nll)
        if key > best_key:
            best_key, best_epoch, stale = key, epoch, 0
            state = {name: value.detach().cpu().clone() for name, value in model.state_dict().items()}
            torch.save({'model_state_dict': state, 'params': params, 'best_epoch': epoch,
                        'val_metrics': val, 'val_nll': nll}, trial_dir / 'best.pt')
        else: stale += 1
        scheduler.step(val['macro_f1'])
        if stale >= PATIENCE: break
    checkpoint = torch.load(trial_dir / 'best.pt', map_location='cpu', weights_only=True)
    result = {'trial': trial_id, **params, 'best_epoch': best_epoch, 'epochs_run': len(history),
              'seconds': time.monotonic() - start, 'val_nll': checkpoint['val_nll'],
              **{f'val_{k}': v for k, v in checkpoint['val_metrics'].items()}}
    del model, optimizer, train_loader, val_loader
    if DEVICE.type == 'cuda': torch.cuda.empty_cache()
    return result

In [ ]:
trials = sample_trials(SEED, NUM_TRIALS)
(RUN_DIR / 'search_plan.json').write_text(json.dumps({'seed': SEED, 'max_epochs': MAX_EPOCHS,
    'patience': PATIENCE, 'selection': 'max validation macro F1, then min validation NLL', 'trials': trials}, indent=2))
results = []
for trial_id, params in enumerate(tqdm(trials, desc='Random search')):
    try:
        result = train_trial(params, trial_id)
        result['status'] = 'complete'
    except RuntimeError as error:
        # GPU 메모리 부족은 기록하고 다른 설정을 시도합니다. 다른 오류는 숨기지 않습니다.
        if 'out of memory' not in str(error).lower(): raise
        if DEVICE.type == 'cuda': torch.cuda.empty_cache()
        result = {'trial': trial_id, **params, 'status': 'failed_oom', 'error': str(error)}
    results.append(result)
    pd.DataFrame(results).to_csv(RUN_DIR / 'trials.csv', index=False)
completed = pd.DataFrame([r for r in results if r['status'] == 'complete'])
assert len(completed), '완료된 실험이 없습니다. 배치 크기를 줄여 새 RUN_DIR에서 실행하세요.'
ranking = completed.sort_values(['val_macro_f1', 'val_nll'], ascending=[False, True])
display(ranking)
best_trial = int(ranking.iloc[0]['trial'])
best_checkpoint = torch.load(RUN_DIR / f'trial_{best_trial:02d}' / 'best.pt', map_location='cpu', weights_only=True)
best_params = best_checkpoint['params']
# 로컬 추론에 필요한 클래스·모델·전처리·설정까지 묶어 저장합니다.
versions = {name: importlib.metadata.version(name) for name in ('torch', 'numpy', 'Pillow', 'scikit-learn', 'matplotlib', 'pandas', 'tqdm')}
metadata = {'model': 'SmallCNN-v1', 'classes': CLASSES, 'image_size': [64, 64],
            'mean': MEAN, 'std': STD, 'crop_total_margin_px': 12, 'resize': 'PIL_BILINEAR',
            'camera': dataset_config['camera'], 'dataset_zip_sha256': archive_sha,
            'manifest_sha256': dataset_config['manifest_sha256'], 'seed': SEED, 'versions': versions,
            'best_trial': best_trial, 'params': best_params, 'best_epoch': best_checkpoint['best_epoch'],
            'validation_metrics': best_checkpoint['val_metrics'], 'validation_nll': best_checkpoint['val_nll'],
            'test_used_for_selection': False}
torch.save({**best_checkpoint, 'metadata': metadata}, RUN_DIR / 'best_model.pt')
(RUN_DIR / 'best_params.json').write_text(json.dumps(metadata, indent=2))
print('best model:', RUN_DIR / 'best_model.pt')

## 탐색·학습 그래프

정확도는 전체 정답 비율, macro F1은 두 종류 각각의 F1을 동등하게 평균한 값입니다. precision은 해당 종류라고 예측한 것 중 정답 비율, recall은 실제 해당 종류 중 찾아낸 비율입니다. ROC-AUC는 임계값을 바꿨을 때 두 종류를 구분하는 능력이며 1에 가까울수록 좋습니다. 손실은 낮을수록 좋지만 과적합 여부는 학습/검증을 함께 봅니다.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for _, row in completed.iterrows():
    history = pd.read_csv(RUN_DIR / f"trial_{int(row['trial']):02d}" / 'history.csv')
    axes[0].plot(history['epoch'], history['val_macro_f1'], label=f"trial {int(row['trial'])}")
axes[0].set_title('Validation macro F1'); axes[0].set_xlabel('epoch'); axes[0].legend(fontsize=7)
axes[1].scatter(completed['lr'], completed['val_macro_f1']); axes[1].set_xscale('log')
axes[1].set_title('Learning rate vs validation F1')
comparison = completed.set_index('trial')[['val_accuracy', 'val_macro_f1', 'val_roc_auc']]
comparison.plot.bar(ax=axes[2], ylim=(0, 1.05), title='Trial comparison')
plt.tight_layout(); plt.savefig(RUN_DIR / 'search_comparison.png'); plt.show()
best_history = pd.read_csv(RUN_DIR / f'trial_{best_trial:02d}' / 'history.csv')
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(best_history['epoch'], best_history['train_loss'], label='train')
axes[0].plot(best_history['epoch'], best_history['val_loss'], label='validation'); axes[0].set_title('Selected loss')
axes[1].plot(best_history['epoch'], best_history['train_accuracy'], label='train')
axes[1].plot(best_history['epoch'], best_history['val_accuracy'], label='validation'); axes[1].set_title('Accuracy')
for field in ('val_macro_f1', 'val_roc_auc', 'val_macro_precision', 'val_macro_recall'):
    axes[2].plot(best_history['epoch'], best_history[field], label=field)
axes[2].set_title('Validation metrics')
for ax in axes: ax.set_xlabel('epoch'); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(RUN_DIR / 'best_learning_curves.png'); plt.show()

## 왜 이 설정을 선택했는가?

아래 그림은 각 파라미터를 사용한 실험들의 **검증 macro F1**을 비교합니다. 빨간 별은 최종 선택된 실험입니다. 학습률·weight decay는 로그 축, 나머지는 설정별 분포를 보여줍니다. 표와 선택 이유도 함께 저장합니다.

여러 파라미터를 동시에 바꾼 랜덤 탐색이므로, 그룹 평균이 높다는 것만으로 그 파라미터 하나가 원인이라고 결론내리지는 않습니다. 최종 선택은 **전체 설정 조합의 최고 검증 F1, 동률이면 최저 검증 NLL**에 근거합니다.

이미 학습한 결과에도 이 셀을 사용할 수 있습니다. `RUN_DIR`를 그 결과 폴더로 지정하면 저장된 `trials.csv`와 `best_params.json`만 읽어 그림을 다시 만듭니다. 학습·테스트를 다시 실행하지 않습니다.

In [ ]:
# 이전 학습 결과에도 쓸 수 있도록 모델/데이터 대신 저장된 실험 기록만 읽습니다.
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

def plot_parameter_evidence(run_dir):
    run_dir = Path(run_dir)
    frame = pd.read_csv(run_dir / 'trials.csv')
    frame = frame[frame['status'] == 'complete'].copy()
    assert len(frame), '완료된 실험 기록이 필요합니다.'
    metadata = json.loads((run_dir / 'best_params.json').read_text())
    selected = int(metadata['best_trial'])
    ranking = frame.sort_values(['val_macro_f1', 'val_nll'], ascending=[False, True])
    best = frame[frame['trial'] == selected].iloc[0]
    # 기록된 선택 기준과 실제 선택 모델이 일치하는지도 확인합니다.
    assert selected in set(ranking[ranking['val_macro_f1'].sub(ranking.iloc[0]['val_macro_f1']).abs() < 1e-12]['trial'])
    columns = ['trial', 'optimizer', 'loss', 'lr', 'batch_size', 'weight_decay', 'dropout',
               'best_epoch', 'val_accuracy', 'val_macro_f1', 'val_roc_auc', 'val_nll']
    ranking[columns].to_csv(run_dir / 'parameter_evidence.csv', index=False)
    display(ranking[columns])
    ties = ranking[ranking['val_macro_f1'].sub(best['val_macro_f1']).abs() < 1e-12]
    reason = f"선택 실험: {selected}, 검증 macro F1={best['val_macro_f1']:.6f}, 공통 NLL={best['val_nll']:.6f}. "
    if len(ties) > 1:
        reason += f"최고 F1이 같은 실험 {len(ties)}개 중 NLL이 가장 낮은 모델을 선택했습니다."
    else:
        reason += '완료된 실험 중 검증 F1이 가장 높아서 선택했습니다.'
    reason += '\n이 결과는 탐색한 설정 조합 안에서의 선택이며, 특정 파라미터 한 개의 효과를 분리한 실험은 아닙니다.'
    (run_dir / 'selection_reason.txt').write_text(reason)
    print(reason)
    fig, axes = plt.subplots(2, 3, figsize=(16, 8))
    # 범주 설정별로 모든 실험 점수와 표본 수를 보여줍니다.
    for ax, column in zip(axes.flat, ['optimizer', 'loss', 'batch_size', 'dropout']):
        groups = [(value, group) for value, group in frame.groupby(column, sort=True)]
        ax.boxplot([group['val_macro_f1'].to_numpy() for _, group in groups], showfliers=False)
        for i, (value, group) in enumerate(groups, start=1):
            ax.scatter([i] * len(group), group['val_macro_f1'], alpha=.6, s=25)
            if value == best[column]:
                ax.scatter(i, best['val_macro_f1'], marker='*', s=180, color='red', zorder=5)
        ax.set_xticks(range(1, len(groups) + 1), [f'{value}\nn={len(group)}' for value, group in groups])
        ax.set_title(column); ax.set_ylim(0, 1.05); ax.set_ylabel('Validation macro F1')
    # 연속 설정은 로그 축으로 비교하고 선택된 조합에 별표를 찍습니다.
    for ax, column in zip(axes.flat[4:], ['lr', 'weight_decay']):
        ax.scatter(frame[column], frame['val_macro_f1'], alpha=.7)
        ax.scatter(best[column], best['val_macro_f1'], marker='*', s=180, color='red', label=f'Best trial {selected}')
        ax.set_xscale('log'); ax.set_xlabel(column); ax.set_ylabel('Validation macro F1')
        ax.set_ylim(0, 1.05); ax.legend()
    fig.suptitle('Hyperparameter comparison (red star = selected combination)')
    plt.tight_layout(); plt.savefig(run_dir / 'parameter_comparison.png'); plt.show()
    return ranking[columns]

parameter_evidence = plot_parameter_evidence(RUN_DIR)


## 최종 테스트: 모델 선택이 끝난 뒤 한 번

아래 셀은 최종 지표·혼동행렬·ROC·오분류 사진과 크기별 성능을 저장합니다. test loss는 모든 모델에 공통인 일반 교차엔트로피(NLL)를 사용합니다. 오분류가 없으면 없다고 표시합니다. 크기별 표본 수가 적은 구간의 성능은 전체 성능으로 일반화하지 마세요.

세션을 다시 열었다면 이전 셀에서 같은 데이터·클래스를 확인한 뒤 `RUN_DIR`를 기존 결과 폴더로 지정하고 저장된 `best_model.pt`로 평가할 수 있습니다. 이미 최종 테스트가 저장된 폴더에서는 재평가를 막습니다.

In [ ]:
assert not (RUN_DIR / 'test_metrics.json').exists(), '이 run의 최종 테스트는 이미 저장됐습니다.'
checkpoint = torch.load(RUN_DIR / 'best_model.pt', map_location='cpu', weights_only=True)
model = SmallCNN(checkpoint['params']['dropout']).to(DEVICE)
model.load_state_dict(checkpoint['model_state_dict'])
test_loader = DataLoader(ShapeDataset(by_split['test']), batch_size=128, shuffle=False, num_workers=0)
test_metrics, y_test, probabilities = evaluate(model, test_loader, nn.CrossEntropyLoss())
predictions = probabilities.argmax(axis=1)
(RUN_DIR / 'test_metrics.json').write_text(json.dumps(test_metrics, indent=2))
report = classification_report(y_test, predictions, labels=[0, 1], target_names=CLASSES, output_dict=True, zero_division=0)
(RUN_DIR / 'classification_report.json').write_text(json.dumps(report, indent=2))
display(pd.DataFrame(report).T)
print('test metrics:', test_metrics)
# 모든 예측을 사진·크기와 연결해 이후 발표와 오분류 분석에 사용합니다.
prediction_rows = []
for row, truth, pred, probability in zip(by_split['test'], y_test, predictions, probabilities):
    prediction_rows.append({**row, 'true_class': CLASSES[int(truth)], 'predicted_class': CLASSES[int(pred)],
                            'correct': bool(truth == pred), 'p_cuboid': float(probability[0]),
                            'p_cylinder': float(probability[1]), 'height_cm': row['size_m'][-1] * 100,
                            'width_cm': max(row['size_m'][:-1]) * 100})
prediction_frame = pd.DataFrame(prediction_rows)
prediction_frame.to_csv(RUN_DIR / 'test_predictions.csv', index=False)
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
ConfusionMatrixDisplay(confusion_matrix(y_test, predictions, labels=[0, 1]), display_labels=CLASSES).plot(ax=axes[0], colorbar=False)
fpr, tpr, _ = roc_curve(y_test, probabilities[:, 1])
axes[1].plot(fpr, tpr, label=f"AUC={test_metrics['roc_auc']:.3f}")
axes[1].plot([0, 1], [0, 1], '--'); axes[1].set_xlabel('False positive rate'); axes[1].set_ylabel('True positive rate')
axes[1].set_title('ROC (positive=cylinder)'); axes[1].legend()
metric_names = ['accuracy', 'macro_f1', 'macro_precision', 'macro_recall', 'roc_auc']
axes[2].bar(metric_names, [test_metrics[name] for name in metric_names]); axes[2].set_ylim(0, 1.05)
axes[2].tick_params(axis='x', rotation=45); axes[2].set_title('Final test scores')
plt.tight_layout(); plt.savefig(RUN_DIR / 'test_scores_confusion_roc.png'); plt.show()
# 높이 구간별 정확도/F1과 표본 수를 함께 보여줍니다.
binned = prediction_frame.assign(height_bin=pd.cut(prediction_frame['height_cm'], [2.99, 4, 5, 6, 7.01]))
size_scores = []
for bucket, group in binned.groupby('height_bin', observed=True):
    truth = [CLASSES.index(s) for s in group['true_class']]
    pred = [CLASSES.index(s) for s in group['predicted_class']]
    size_scores.append({'height_cm': str(bucket), 'images': len(group), 'accuracy': accuracy_score(truth, pred),
                        'macro_f1': f1_score(truth, pred, labels=[0, 1], average='macro', zero_division=0)})
size_frame = pd.DataFrame(size_scores); size_frame.to_csv(RUN_DIR / 'height_scores.csv', index=False)
display(size_frame)
size_frame.set_index('height_cm')[['accuracy', 'macro_f1']].plot.bar(ylim=(0, 1.05), title='Scores by height')
plt.tight_layout(); plt.savefig(RUN_DIR / 'height_scores.png'); plt.show()
wrong = prediction_frame[~prediction_frame['correct']].copy()
wrong['confidence'] = wrong[['p_cuboid', 'p_cylinder']].max(axis=1)
wrong = wrong.sort_values('confidence', ascending=False)
wrong.to_csv(RUN_DIR / 'misclassified.csv', index=False)
if wrong.empty:
    print('이 테스트 분할에서는 오분류가 없습니다. 실제 조건 전체에서 오류가 없다는 뜻은 아닙니다.')
else:
    shown = wrong.head(16)
    fig, axes = plt.subplots(math.ceil(len(shown) / 4), 4, figsize=(12, 3 * math.ceil(len(shown) / 4)), squeeze=False)
    for ax in axes.flat: ax.axis('off')
    for ax, (_, row) in zip(axes.flat, shown.iterrows()):
        with Image.open(DATA_ROOT / row['path']) as image: ax.imshow(image)
        ax.set_title(f"true: {row['true_class']}\npred: {row['predicted_class']} ({row['confidence']:.2f})\nh={row['height_cm']:.1f}cm", fontsize=9)
    plt.tight_layout(); plt.savefig(RUN_DIR / 'misclassified.png'); plt.show()
print('결과 저장 폴더:', RUN_DIR)

## 저장 결과와 다음 단계

- `best_model.pt`: 최고 검증 epoch의 가중치와 추론용 설정
- `best_params.json`: 선택한 하이퍼파라미터·클래스·정규화·카메라·데이터 해시·패키지 버전
- `trials.csv`, `trial_XX/history.csv`: 실험별 비교와 epoch별 기록
- `*.png`: 데이터 분포·학습 곡선·실험 비교·최종 지표·혼동행렬·ROC·오분류 사진
- `test_predictions.csv`, `misclassified.csv`: 어떤 사진을 어떻게 예측했는지

`best_model.pt`와 `best_params.json`을 Ubuntu의 `checkpoints/`에 가져옵니다. 이후 CNN 로컬 추론·혼합 종류 운반을 연결합니다. 다양한 크기의 사진 분류 성공은 그 크기의 파지·운반 성공과 별개의 결과입니다. 중단 후에는 새 run 폴더에서 시작하세요. 완료된 trial의 가중치·기록은 Drive에 남습니다.

공식 참고: [PyTorch CrossEntropyLoss](https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html), [PyTorch optimizers](https://docs.pytorch.org/docs/stable/optim.html), [scikit-learn ROC-AUC](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.roc_auc_score.html).